In [2]:
%uv pip install -q bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [5]:
import subprocess, sys, torch

# Flash Attention 2 requires a pre-built wheel — pip source build often fails
# We install the correct pre-compiled wheel for your CUDA + torch version

def get_flash_attn_wheel():
    cuda_version = torch.version.cuda.replace(".", "")   # e.g. "121"
    torch_version = torch.__version__.split("+")[0].replace(".", "")[:3]  # e.g. "213"
    python_version = f"cp{sys.version_info.major}{sys.version_info.minor}"  # e.g. "cp311"
    
    print(f"  Detected: Python={python_version}, Torch={torch.__version__}, CUDA={torch.version.cuda}")
    return None  # signal to use pip directly

get_flash_attn_wheel()

print("Installing flash-attn (this may take 3–5 min on first run)...")
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "flash-attn",
        "--no-build-isolation",   # uses your existing torch headers
        "-q",
    ],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✓ flash-attn installed successfully.")
else:
    print("✗ flash-attn install failed. Falling back to eager attention.")
    print("  stderr:", result.stderr[-500:])  # last 500 chars of error
    # Patch so Cell 1's graceful fallback triggers cleanly
    import builtins
    _real_import = builtins.__import__
    def _patched_import(name, *args, **kwargs):
        if name == "flash_attn":
            raise ImportError("flash_attn not available — using eager attention")
        return _real_import(name, *args, **kwargs)
    builtins.__import__ = _patched_import

  Detected: Python=cp312, Torch=2.8.0+cu129, CUDA=12.9
Installing flash-attn (this may take 3–5 min on first run)...
✓ flash-attn installed successfully.


In [6]:
import os
import pandas as pd

base_path = '/root/physionet.org/files/mimic-iv-note/2.2/note'

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)

            print("=" * 80)
            print(f"FILE: {file_path}")

            try:
                df = pd.read_csv(file_path)

                # First 3 rows
                print("\n--- HEAD (first 3 rows) ---")
                display(df.head(3))

                # Info
                print("\n--- INFO ---")
                df.info()

                # Statistics
                print("\n--- DESCRIBE (all columns) ---")
                display(df.describe(include='all'))

            except Exception as e:
                print(f"Error reading {file_path}: {e}")

FILE: /root/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv

--- HEAD (first 3 rows) ---


,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,\nName: ___ Unit No: _...



--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331793 entries, 0 to 331792
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   note_id     331793 non-null  object
 1   subject_id  331793 non-null  int64 
 2   hadm_id     331793 non-null  int64 
 3   note_type   331793 non-null  object
 4   note_seq    331793 non-null  int64 
 5   charttime   331793 non-null  object
 6   storetime   331776 non-null  object
 7   text        331793 non-null  object
dtypes: int64(3), object(5)
memory usage: 20.3+ MB

--- DESCRIBE (all columns) ---


,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
count,331793,3.317930e+05,3.317930e+05,331793,331793.000000,331793,331776,331793
unique,331793,NaN,NaN,1,NaN,35387,329891,331792
top,19999987-DS-2,NaN,NaN,DS,NaN,2129-06-10 00:00:00,2150-08-22 21:54:00,\nName: ___ Unit No: ___...
freq,1,NaN,NaN,331793,NaN,25,3,2
mean,NaN,1.501168e+07,2.499928e+07,NaN,15.177825,NaN,NaN,NaN
std,NaN,2.880278e+06,2.889827e+06,NaN,9.417676,NaN,NaN,NaN
min,NaN,1.000003e+07,2.000002e+07,NaN,2.000000,NaN,NaN,NaN
25%,NaN,1.251681e+07,2.249425e+07,NaN,9.000000,NaN,NaN,NaN
50%,NaN,1.500848e+07,2.500190e+07,NaN,14.000000,NaN,NaN,NaN
75%,NaN,1.751464e+07,2.750393e+07,NaN,20.000000,NaN,NaN,NaN


FILE: /root/physionet.org/files/mimic-iv-note/2.2/note/discharge_detail.csv

--- HEAD (first 3 rows) ---


...

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
count,2321355,2.321355e+06,1.144758e+06,2321355,2.321355e+06,2321355,2321355,2321355
unique,2321355,NaN,NaN,2,NaN,2149546,2201003,2299451
top,19999987-RR-21,NaN,NaN,RR,NaN,2145-05-06 14:17:00,2146-07-03 10:10:00,EXAMINATION: BILATERAL DIGITAL SCREENING MAMM...
freq,1,NaN,NaN,2295635,NaN,7,12,285
mean,NaN,1.500652e+07,2.500383e+07,NaN,4.396245e+01,NaN,NaN,NaN
std,NaN,2.883818e+06,2.884478e+06,NaN,4.312100e+01,NaN,NaN,NaN
min,NaN,1.000003e+07,2.000002e+07,NaN,2.000000e+00,NaN,NaN,NaN
25%,NaN,1.250505e+07,2.250513e+07,NaN,1.800000e+01,NaN,NaN,NaN
50%,NaN,1.501340e+07,2.500701e+07,NaN,3.000000e+01,NaN,NaN,NaN
75%,NaN,1.750020e+07,2.749573e+07,NaN,5.500000e+01,NaN,NaN,NaN


FILE: /root/physionet.org/files/mimic-iv-note/2.2/note/radiology_detail.csv

--- HEAD (first 3 rows) ---


,note_id,subject_id,field_name,field_value,field_ordinal
0,10000032-RR-14,10000032,exam_code,C11,1
1,10000032-RR-14,10000032,exam_name,CHEST (PA & LAT),1
2,10000032-RR-15,10000032,exam_code,U314,1



--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6046121 entries, 0 to 6046120
Data columns (total 5 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   note_id        object
 1   subject_id     int64 
 2   field_name     object
 3   field_value    object
 4   field_ordinal  int64 
dtypes: int64(2), object(3)
memory usage: 230.6+ MB

--- DESCRIBE (all columns) ---


,note_id,subject_id,field_name,field_value,field_ordinal
count,6046121,6.046121e+06,6046121,6046121,6.046121e+06
unique,2327290,NaN,5,56423,NaN
top,15349002-RR-77,NaN,exam_code,CHEST (PORTABLE AP),NaN
freq,271,NaN,2913024,334955,NaN
mean,NaN,1.500725e+07,NaN,NaN,1.564501e+00
std,NaN,2.883773e+06,NaN,NaN,2.368526e+00
min,NaN,1.000003e+07,NaN,NaN,1.000000e+00
25%,NaN,1.250262e+07,NaN,NaN,1.000000e+00
50%,NaN,1.501590e+07,NaN,NaN,1.000000e+00
75%,NaN,1.750090e+07,NaN,NaN,1.000000e+00


In [7]:
# ═══ OOM DIAGNOSTIC PROBE ════════════════════════════════════════════
# Run this as a standalone cell. No training. Just measures what fits.
%uv pip install peft
import os, gc, time, math, traceback
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL = "BioMistral/BioMistral-7B"
DTYPE = torch.bfloat16

def reset_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def vram_snapshot(label):
    a = torch.cuda.memory_allocated() / 1e9
    r = torch.cuda.memory_reserved() / 1e9
    p = torch.cuda.max_memory_allocated() / 1e9
    print(f"  [{label:30s}] alloc={a:6.2f} GB | reserved={r:6.2f} GB | peak={p:6.2f} GB")

# ── 1. GPU inventory ──────────────────────────────────────────────────
print("=" * 72)
print("GPU INVENTORY")
print("=" * 72)
if torch.cuda.is_available():
    print(f"  Device:         {torch.cuda.get_device_name(0)}")
    print(f"  Total VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"  bf16 supported: {torch.cuda.is_bf16_supported()}")
    print(f"  Torch version:  {torch.__version__}")

# ── 2. Attention backends available ───────────────────────────────────
print("\n" + "=" * 72)
print("ATTENTION BACKENDS AVAILABLE")
print("=" * 72)
try:
    import flash_attn
    print(f"  ✓ flash_attn {flash_attn.__version__} importable")
except Exception as e:
    print(f"  ✗ flash_attn NOT importable: {e}")

try:
    from torch.nn.functional import scaled_dot_product_attention
    print(f"  ✓ torch SDPA available (built-in memory-efficient attention)")
except Exception as e:
    print(f"  ✗ SDPA NOT available: {e}")

# ── 3. Probe model loading + one fwd/bwd at several configs ───────────
# We try the SMALLEST config first so you at least see a number even if
# larger ones fail. Each trial is in isolation with full VRAM reset.

def trial(attn_impl, batch, seq, lora_r=32):
    print(f"\n  ── Trial: attn={attn_impl}, batch={batch}, seq={seq}, lora_r={lora_r} ──")
    reset_vram()
    try:
        tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, device_map="auto", trust_remote_code=True,
            torch_dtype=DTYPE, attn_implementation=attn_impl,
        )
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        vram_snapshot("after base load")

        cfg_lora = LoraConfig(
            r=lora_r, lora_alpha=lora_r * 2, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            bias="none", task_type=TaskType.CAUSAL_LM,
        )
        model = get_peft_model(model, cfg_lora)
        vram_snapshot("after LoRA wrap")

        # Fake batch
        ids = torch.randint(0, 32000, (batch, seq), device="cuda", dtype=torch.long)
        attn_mask = torch.ones_like(ids)
        labels = ids.clone()

        optim = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=1e-4, fused=True,
        )

        t0 = time.time()
        with torch.autocast("cuda", dtype=DTYPE):
            out = model(input_ids=ids, attention_mask=attn_mask, labels=labels)
        vram_snapshot("after forward")

        out.loss.backward()
        vram_snapshot("after backward")

        optim.step()
        vram_snapshot("after optim step")
        print(f"  ✓ SUCCESS — {time.time()-t0:.2f}s")

        # Clean
        del model, optim, ids, attn_mask, labels, out
        reset_vram()
        return True

    except torch.cuda.OutOfMemoryError as e:
        print(f"  ✗ OOM: {str(e)[:150]}")
        try:
            del model
        except: pass
        reset_vram()
        return False
    except Exception as e:
        print(f"  ✗ ERROR: {type(e).__name__}: {str(e)[:200]}")
        traceback.print_exc(limit=3)
        try:
            del model
        except: pass
        reset_vram()
        return False

# ── 4. Run trials — from safe to aggressive ──────────────────────────
# Confirm flash-attn works at a small config first, then scale up.
# Adjust or comment out rows to stop early.

results = {}
trials = [
    ("flash_attention_2", 1, 2048),
    ("flash_attention_2", 2, 4096),
    ("flash_attention_2", 4, 4096),
    ("flash_attention_2", 8, 4096),
    ("flash_attention_2", 16, 4096),
    ("flash_attention_2", 24, 4096),     # your friend's setting
    # uncomment to confirm eager is indeed the killer:
    # ("eager", 1, 4096),
    # ("eager", 1, 2048),
]
print("\n" + "=" * 72)
print("RUNNING TRIALS")
print("=" * 72)
for attn, b, s in trials:
    ok = trial(attn, b, s)
    results[(attn, b, s)] = ok

print("\n" + "=" * 72)
print("SUMMARY — largest working batch per (attn, seq)")
print("=" * 72)
for (attn, b, s), ok in results.items():
    mark = "✓" if ok else "✗"
    print(f"  {mark}  attn={attn:22s}  batch={b:3d}  seq={s:4d}")

Using Python 3.12.6 environment at: /usr/local
Resolved 43 packages in 284ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 108ms
Installed 1 package in 349ms
 + peft==0.19.1
Note: you may need to restart the kernel to use updated packages.
GPU INVENTORY
  Device:         NVIDIA H200
  Total VRAM:     150.1 GB
  bf16 supported: True
  Torch version:  2.8.0+cu129

ATTENTION BACKENDS AVAILABLE
  ✓ flash_attn 2.8.3 importable
  ✓ torch SDPA available (bu

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

  [after base load               ] alloc= 14.48 GB | reserved= 14.49 GB | peak= 14.48 GB
  [after LoRA wrap               ] alloc= 14.83 GB | reserved= 14.84 GB | peak= 14.83 GB
  [after forward                 ] alloc= 32.29 GB | reserved= 32.91 GB | peak= 32.57 GB
  [after backward                ] alloc= 15.41 GB | reserved= 33.17 GB | peak= 32.81 GB
  [after optim step              ] alloc= 16.09 GB | reserved= 33.46 GB | peak= 32.81 GB
  ✓ SUCCESS — 1.74s

  ── Trial: attn=flash_attention_2, batch=2, seq=4096, lora_r=32 ──
  [after base load               ] alloc= 14.55 GB | reserved= 14.55 GB | peak= 14.55 GB
  [after LoRA wrap               ] alloc= 14.89 GB | reserved= 14.91 GB | peak= 14.89 GB
  [after forward                 ] alloc= 84.10 GB | reserved= 86.62 GB | peak= 85.21 GB
  [after backward                ] alloc= 15.89 GB | reserved= 87.67 GB | peak= 86.20 GB
  [after optim step              ] alloc= 16.57 GB | reserved= 87.84 GB | peak= 86.20 GB
  ✓ SUCCESS — 1.10s



In [9]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  BioMistral-7B — Continued Pretraining on Raw MIMIC-IV Notes  (CPT v3)     ║
║  Hardware: NVIDIA H200 (~150 GB VRAM)                                       ║
║  Time budget: 5 hours — deadline build                                      ║
║  Output:   /mnt/biomistral                                                  ║
╚══════════════════════════════════════════════════════════════════════════════╝

WHY v3 (what changed from v2)
──────────────────────────────────────────────────────────────────────────
v2 was sized for a 22h run over 2B tokens. Budget is now 5h. The big
insight from v2's live log: throughput is 6,817 tok/s (better than
estimated), so we're not compute-bound, we're SCHEDULE-bound. With
grad_accum=24, each optimizer step takes ~29s, meaning only ~600 steps
fit in 5h. But the cosine schedule was configured for 10,642 steps —
LR barely decayed inside our window, wasting most of the training signal.

v3 fixes this by:
  1. grad_accum     24 → 8       (3× more optim steps, smaller effective batch
                                   but more LR updates — better for short runs)
  2. Schedule total 10,642 → 900  (cosine actually decays within our window)
  3. warmup_ratio   0.03 → 0.05   (45 warmup steps vs 27 at 900 total)
  4. eval_every     200  → 50     (tighter tracking of short curve)
  5. save_every     1000 → 150    (don't lose progress on preempt)
  6. wall-clock     22h  → 4.5h   (hard guarantee on clean exit w/ plots)
  7. Early stop patience 5 → 4    (bail faster if plateau)

KEPT from v2:
  - batch=2 (probed ceiling on H200)
  - max_seq_len=4096
  - lora_r=16, alpha=32, dropout=0.10 (anti-overfit stack)
  - forced flash-attn with assert
  - random window stride offset per epoch
  - resume-from-checkpoint support

EXPECTED TIMELINE (5h budget)
──────────────────────────────────────────────────────────────────────────
  900 optim steps × 8 accum × ~1.3s/micro  ≈ 2.6h  pure training
  18 evals × ~30s                          ≈ 9m
  6 checkpoints × ~5s                      ≈ 30s
  Final eval (full val)                    ≈ 5m
  Plots + summary                          ≈ 1m
  TOTAL                                    ≈ 2.9h  (buffer to 4.5h hard cap)

EFFECTIVE TOKENS SEEN
──────────────────────────────────────────────────────────────────────────
  900 steps × 16 eff batch × 4096 seq = 59M tokens
  This is ~3% of the full corpus but sampled uniformly from 510K windows
  = tens of thousands of distinct notes. Sufficient for CPT to shift
  the model's domain distribution (goal: reduce val_ppl from base to ~2.5).
"""

# ═══ CELL 1 — Dependencies ════════════════════════════════════════════════

import subprocess, sys

_pkgs = [
    "transformers>=4.40", "bitsandbytes>=0.43", "peft>=0.10",
    "accelerate>=0.30", "sentencepiece", "scipy", "scikit-learn",
    "matplotlib",
]
for pkg in _pkgs:
    name = pkg.split(">=")[0].replace("-", "_")
    try:
        __import__(name)
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except subprocess.CalledProcessError:
            print(f"  ⚠ Could not install {pkg}")

print("✓ Dependencies ready.")


# ═══ CELL 2 — Imports & Config ═══════════════════════════════════════════════

import os, re, json, warnings, time, gc, math, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig, get_peft_model, TaskType, PeftModel,
)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")


class Config:
    # ══════════════════════════════════════════════════════════════════════
    # PATHS
    # ══════════════════════════════════════════════════════════════════════
    note_dir       = r"/root/physionet.org/files/mimic-iv-note/2.2/note"
    discharge_file = r"/root/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv"
    radiology_file = r"/root/physionet.org/files/mimic-iv-note/2.2/note/radiology.csv"
    model_name     = "BioMistral/BioMistral-7B"

    output_dir     = "/mnt/biomistral"
    cache_dir      = "/mnt/biomistral/cache"
    ckpt_dir       = "/mnt/biomistral/checkpoints"
    plot_dir       = "/mnt/biomistral/plots"
    best_dir       = "/mnt/biomistral/best_model"
    final_dir      = "/mnt/biomistral/final_model"
    log_dir        = "/mnt/biomistral/logs"
    resume_state   = "/mnt/biomistral/checkpoints/resume_state.pt"

    # ══════════════════════════════════════════════════════════════════════
    # DATA — only used if cache isn't already built
    # ══════════════════════════════════════════════════════════════════════
    min_note_chars     = 100
    max_note_chars     = 60_000
    csv_chunk_size     = 20_000
    val_subject_frac   = 0.005
    seed               = 42
    use_headers        = True

    # ══════════════════════════════════════════════════════════════════════
    # PACKING
    # ══════════════════════════════════════════════════════════════════════
    max_seq_len        = 4096
    use_cache          = True
    # v2: random stride offset each epoch, breaks position-bias
    randomize_window_offset = True

    # ══════════════════════════════════════════════════════════════════════
    # MODEL / LoRA  — v2: halved rank, doubled dropout (anti-overfit)
    # ══════════════════════════════════════════════════════════════════════
    lora_r             = 16          # v2: 32 → 16
    lora_alpha         = 32          # v2: 64 → 32 (keep ratio)
    lora_dropout       = 0.10        # v2: 0.05 → 0.10
    lora_targets       = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"]
    use_grad_ckpt      = True

    # ══════════════════════════════════════════════════════════════════════
    # TRAINING — v3: 5-hour budget. grad_accum 24→8 for 3× more optim steps.
    # Effective batch 48→16. Smaller batch = more LR updates per unit time,
    # which matters more than batch size for short CPT runs.
    # ══════════════════════════════════════════════════════════════════════
    batch_size         = 2           # probed ceiling on H200
    grad_accum_steps   = 8           # v3: 24→8 (3× more optim steps)
    lr                 = 2e-4
    min_lr_ratio       = 0.05
    weight_decay       = 0.01
    warmup_ratio       = 0.05        # v3: 0.03→0.05 (short run needs faster warmup)
    num_epochs         = 1
    max_grad_norm      = 1.0
    use_8bit_optim     = False

    # v3: Cap total optimizer steps so cosine schedule fully anneals in 5h.
    # Set to None to fall back to (len(loader) // grad_accum) * num_epochs.
    # At ~29s/step with accum=24 that was 10,642 steps / 86h — cosine barely
    # moved. With accum=8 we target ~900 steps and the cosine actually cycles.
    max_optimizer_steps = 900

    # Logging / eval / save — v3: tighter cadence for short run
    log_every_steps    = 10          # v3: 20→10
    eval_every_steps   = 50          # v3: 200→50
    save_every_steps   = 150         # v3: 1000→150
    keep_last_n_ckpts  = 3

    # Eval batches
    eval_max_batches_mid   = 30      # v3: 50→30 (faster mid-train evals)
    eval_max_batches_final = None    # full val set at end

    # Early stop
    early_stop_patience = 4          # v3: 5→4 (bail faster)

    # Wall-clock — v3: 4.5h hard cap, leaves buffer for final eval + plots
    max_training_seconds = int(4.5 * 3600)


cfg = Config()
for d in [cfg.output_dir, cfg.cache_dir, cfg.ckpt_dir, cfg.plot_dir,
          cfg.best_dir, cfg.final_dir, cfg.log_dir]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Startup banner ──
print("=" * 72)
print("BioMistral-7B CPT on MIMIC-IV Notes — v3 (5-hour deadline build)")
print("=" * 72)
print(f"  Device:          {device}")
gpu_name = "cpu"
vram_gb  = 0.0
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU:             {gpu_name} | VRAM: {vram_gb:.1f} GB")
print(f"  Output:          {cfg.output_dir}")
print(f"  Model:           {cfg.model_name}")
print(f"  LoRA:            r={cfg.lora_r}, alpha={cfg.lora_alpha}, "
      f"dropout={cfg.lora_dropout} (anti-overfit)")
print(f"  Seq len:         {cfg.max_seq_len}  (packed)")
print(f"  Batch:           {cfg.batch_size} × {cfg.grad_accum_steps} accum = "
      f"{cfg.batch_size * cfg.grad_accum_steps} effective")
print(f"  LR:              {cfg.lr:.1e} cosine → {cfg.lr*cfg.min_lr_ratio:.1e}")
print(f"  Early stop:      patience={cfg.early_stop_patience} evals on val_ppl")
print(f"  Wall-clock cap:  {cfg.max_training_seconds/3600:.0f} h")
print("=" * 72)


def print_vram(label=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1e9
        r = torch.cuda.memory_reserved() / 1e9
        p = torch.cuda.max_memory_allocated() / 1e9
        print(f"  [VRAM] {label}: alloc={a:.2f} / reserved={r:.2f} / peak={p:.2f} GB")


def force_gc(label=""):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if label:
        print(f"  [GC] {label}")


# ═══ CELL 3 — Text Cleaning & Dedup ══════════════════════════════════════════

_WS_RE        = re.compile(r"[ \t]+")
_MULTI_NL_RE  = re.compile(r"\n{4,}")
_FORM_FEED_RE = re.compile(r"\f")

def clean_note(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = _FORM_FEED_RE.sub("\n", text)
    text = _WS_RE.sub(" ", text)
    text = _MULTI_NL_RE.sub("\n\n\n", text)
    return text.strip()


DEDUP_HASH_CHARS = 512

def content_hash(text: str) -> str:
    if not isinstance(text, str):
        return ""
    prefix = text[:DEDUP_HASH_CHARS].strip().lower()
    return hashlib.md5(prefix.encode("utf-8", errors="ignore")).hexdigest()


def make_header(note_type: str, charttime) -> str:
    nt = (note_type or "NOTE").upper().strip()[:16]
    ts = ""
    if pd.notna(charttime) and charttime:
        ts = str(charttime).strip()
    return f"[{nt} | {ts}]\n" if ts else f"[{nt}]\n"


# ═══ CELL 4 — Tokenizer ═════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("LOADING TOKENIZER")
print("=" * 72)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
EOS_ID = tokenizer.eos_token_id
BOS_ID = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else EOS_ID
print(f"  Vocab: {tokenizer.vocab_size:,}  BOS={BOS_ID} EOS={EOS_ID} PAD={tokenizer.pad_token_id}")


# ═══ CELL 5 — Token Cache (reuses v1's cache if present) ═══════════════════

CACHE_TRAIN_TOKENS = os.path.join(cfg.cache_dir, "train_tokens.bin")
CACHE_VAL_TOKENS   = os.path.join(cfg.cache_dir, "val_tokens.bin")
CACHE_META         = os.path.join(cfg.cache_dir, "cache_meta.json")


def cache_exists() -> bool:
    return (cfg.use_cache and
            os.path.exists(CACHE_TRAIN_TOKENS) and
            os.path.exists(CACHE_VAL_TOKENS) and
            os.path.exists(CACHE_META))


def _append_bin(path, arrays):
    if not arrays:
        return
    cat = np.concatenate(arrays)
    mode = "ab" if os.path.exists(path) else "wb"
    with open(path, mode) as f:
        f.write(cat.tobytes())


def build_token_cache():
    print("\n" + "=" * 72)
    print("BUILDING TOKEN CACHE (streaming) — this is the one-time 45-min pass")
    print("=" * 72)

    print("  Scanning subject_ids for split...")
    all_subjects = set()
    for chunk in pd.read_csv(cfg.discharge_file, usecols=["subject_id"],
                             chunksize=cfg.csv_chunk_size):
        all_subjects.update(chunk["subject_id"].unique().tolist())
    if os.path.exists(cfg.radiology_file):
        for chunk in pd.read_csv(cfg.radiology_file, usecols=["subject_id"],
                                 chunksize=cfg.csv_chunk_size):
            all_subjects.update(chunk["subject_id"].unique().tolist())
    all_subjects = sorted(all_subjects)
    rng = np.random.RandomState(cfg.seed)
    rng.shuffle(all_subjects)
    n_val_subj = max(50, int(len(all_subjects) * cfg.val_subject_frac))
    val_subjects = set(all_subjects[:n_val_subj])
    print(f"  Total subjects: {len(all_subjects):,}")
    print(f"  Val subjects:   {n_val_subj:,} ({100*cfg.val_subject_frac:.2f}%)")

    train_chunks, val_chunks = [], []
    seen_hashes = set()
    stats = {f"{k}_{s}": 0 for k in ["discharge", "radiology"]
             for s in ["rows", "kept", "skipped", "deduped"]}
    stats["train_notes"] = 0
    stats["val_notes"]   = 0
    t_start = time.time()

    def process_csv(path, kind):
        if not os.path.exists(path):
            print(f"  ⚠ {path} not found — skipping")
            return
        print(f"\n  Processing {kind}: {path}")
        cols = ["subject_id", "hadm_id", "note_type", "charttime", "text"]
        for i, chunk in enumerate(pd.read_csv(
            path, usecols=cols, low_memory=False,
            chunksize=cfg.csv_chunk_size,
        )):
            stats[f"{kind}_rows"] += len(chunk)
            for _, row in chunk.iterrows():
                text = row.get("text", "")
                if not isinstance(text, str):
                    stats[f"{kind}_skipped"] += 1
                    continue
                txt = clean_note(text)
                if len(txt) < cfg.min_note_chars or len(txt) > cfg.max_note_chars:
                    stats[f"{kind}_skipped"] += 1
                    continue
                h = content_hash(txt)
                if h in seen_hashes:
                    stats[f"{kind}_deduped"] += 1
                    continue
                seen_hashes.add(h)

                full = (make_header(kind, row.get("charttime", "")) + txt
                        if cfg.use_headers else txt)
                ids = tokenizer.encode(full, add_special_tokens=False)
                ids = [BOS_ID] + ids + [EOS_ID]
                arr = np.asarray(ids, dtype=np.uint32)

                subj = row.get("subject_id", -1)
                if subj in val_subjects:
                    val_chunks.append(arr)
                    stats["val_notes"] += 1
                else:
                    train_chunks.append(arr)
                    stats["train_notes"] += 1
                stats[f"{kind}_kept"] += 1

            elapsed = time.time() - t_start
            tr = sum(len(x) for x in train_chunks)
            vl = sum(len(x) for x in val_chunks)
            print(f"    chunk {i+1:>4} | rows={stats[f'{kind}_rows']:>9,} "
                  f"| kept={stats[f'{kind}_kept']:>8,} "
                  f"| train≈{tr/1e6:>6.1f}M | val≈{vl/1e6:>5.2f}M "
                  f"| {elapsed/60:.1f}m")

            if len(train_chunks) > 10_000:
                _append_bin(CACHE_TRAIN_TOKENS, train_chunks)
                train_chunks.clear()
                force_gc()
            if len(val_chunks) > 5_000:
                _append_bin(CACHE_VAL_TOKENS, val_chunks)
                val_chunks.clear()

    process_csv(cfg.discharge_file, "discharge")
    process_csv(cfg.radiology_file, "radiology")
    if train_chunks: _append_bin(CACHE_TRAIN_TOKENS, train_chunks)
    if val_chunks:   _append_bin(CACHE_VAL_TOKENS, val_chunks)

    n_train = os.path.getsize(CACHE_TRAIN_TOKENS) // 4
    n_val   = os.path.getsize(CACHE_VAL_TOKENS)   // 4
    meta = {
        "n_train_tokens": int(n_train), "n_val_tokens": int(n_val),
        "n_train_notes":  stats["train_notes"],
        "n_val_notes":    stats["val_notes"],
        "stats":          stats,
        "val_subject_frac": cfg.val_subject_frac,
        "max_seq_len":    cfg.max_seq_len,
        "model_name":     cfg.model_name,
        "use_headers":    cfg.use_headers,
    }
    with open(CACHE_META, "w") as f:
        json.dump(meta, f, indent=2, default=str)

    print(f"\n  ✓ Cache built: train={n_train/1e6:.1f}M tok, val={n_val/1e6:.2f}M tok")
    return meta


if cache_exists():
    print("\n  ✓ Token cache found — skipping rebuild.")
    with open(CACHE_META) as f:
        meta = json.load(f)
else:
    for p in [CACHE_TRAIN_TOKENS, CACHE_VAL_TOKENS, CACHE_META]:
        if os.path.exists(p):
            os.remove(p)
    meta = build_token_cache()

n_train_tokens = meta["n_train_tokens"]
n_val_tokens   = meta["n_val_tokens"]
print(f"\n  Using cache: train={n_train_tokens/1e6:.1f}M tok, "
      f"val={n_val_tokens/1e6:.2f}M tok, "
      f"train_notes={meta['n_train_notes']:,}, val_notes={meta['n_val_notes']:,}")


# ═══ CELL 6 — Packed Dataset (with random stride offset per epoch) ══════════

class PackedTokenDataset(Dataset):
    """Token memmap → fixed-length packed windows.

    v2 addition: `offset` parameter lets us shift the window start each
    epoch, so notes get packed at different phases across epochs. This
    prevents the model from over-fitting to any specific note-to-window
    alignment.
    """
    def __init__(self, path: str, seq_len: int, offset: int = 0):
        size = os.path.getsize(path) // 4
        self.mm   = np.memmap(path, dtype=np.uint32, mode="r", shape=(size,))
        self.size = size
        self.seq_len = seq_len
        self.offset  = offset
        self.n_windows = max(0, (size - seq_len - offset) // seq_len)

    def set_offset(self, offset: int):
        self.offset = offset
        self.n_windows = max(0, (self.size - self.seq_len - offset) // self.seq_len)

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx):
        start = self.offset + idx * self.seq_len
        end = start + self.seq_len
        ids = torch.from_numpy(np.array(self.mm[start:end], dtype=np.int64))
        return {
            "input_ids": ids,
            "labels":    ids.clone(),
            "attention_mask": torch.ones(self.seq_len, dtype=torch.long),
        }


train_ds = PackedTokenDataset(CACHE_TRAIN_TOKENS, cfg.max_seq_len)
val_ds   = PackedTokenDataset(CACHE_VAL_TOKENS,   cfg.max_seq_len)
print(f"\n  Packed train windows: {len(train_ds):,}")
print(f"  Packed val windows:   {len(val_ds):,}")


def collate_packed(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "labels":         torch.stack([b["labels"]         for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
    }


# ═══ CELL 7 — Model Load (FORCE flash-attn — no silent fallback) ════════════

print("\n" + "=" * 72)
print("LOADING BioMistral-7B (bf16 + Flash Attention 2 + LoRA r=16)")
print("=" * 72)

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"  Compute dtype: {compute_dtype}")

# HARD REQUIREMENT — no eager fallback. Friend's "eager" edit caused OOM.
try:
    import flash_attn
    print(f"  ✓ flash_attn {flash_attn.__version__} imported")
except ImportError as e:
    raise RuntimeError(
        "flash_attn is REQUIRED for this config. Eager attention needs "
        "~1 TB VRAM at batch=2 seq=4096 and will cause silent OOM crashes. "
        "Install with: pip install flash-attn --no-build-isolation"
    ) from e

attn_impl = "flash_attention_2"
print(f"  Attention:     {attn_impl} (forced)")

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
    attn_implementation=attn_impl,
)
model.config.use_cache = False

# Verify flash-attn actually activated (HF will silently fall back to eager
# if something's off with the install). Print and assert.
actual_attn = getattr(model.config, "_attn_implementation", "unknown")
print(f"  config._attn_implementation = {actual_attn}")
assert actual_attn == "flash_attention_2", (
    f"Flash attn requested but HF loaded {actual_attn}. "
    f"Check flash-attn install. Refusing to continue (will OOM)."
)

if cfg.use_grad_ckpt:
    model.gradient_checkpointing_enable()
print(f"  Base loaded in {time.time()-t0:.1f}s")
print_vram("after base")

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.lora_targets,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"  Total params:  {total_p:>14,}")
print(f"  Trainable:     {trainable:>14,} ({100*trainable/total_p:.2f}%)")
print_vram("after LoRA")


# ═══ CELL 8 — Optimizer + Scheduler ══════════════════════════════════════════

if cfg.use_8bit_optim:
    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.AdamW8bit(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg.lr, weight_decay=cfg.weight_decay,
            betas=(0.9, 0.95), eps=1e-8,
        )
        print("  ✓ 8-bit AdamW (bitsandbytes)")
    except ImportError:
        print("  ⚠ bitsandbytes missing — falling back to fused AdamW")
        cfg.use_8bit_optim = False

if not cfg.use_8bit_optim:
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr, weight_decay=cfg.weight_decay,
        fused=torch.cuda.is_available(),
        betas=(0.9, 0.95), eps=1e-8,
    )
    print("  ✓ Fused AdamW")


train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    collate_fn=collate_packed, num_workers=0, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    collate_fn=collate_packed, num_workers=0, pin_memory=True, drop_last=False,
)

steps_per_epoch = len(train_loader) // cfg.grad_accum_steps
natural_total   = steps_per_epoch * cfg.num_epochs

# v3: cap total_steps so cosine schedule fully anneals within our time budget.
# Without this cap, schedule targets 10K+ steps but we only reach ~900, so LR
# never decays and we train at near-peak LR the whole time (wastes signal).
if cfg.max_optimizer_steps is not None and cfg.max_optimizer_steps < natural_total:
    total_steps = cfg.max_optimizer_steps
    print(f"\n  v3: capping total_steps to {total_steps} "
          f"(natural would be {natural_total:,}) — cosine will fully anneal.")
else:
    total_steps = natural_total

warmup_steps = max(20, int(total_steps * cfg.warmup_ratio))
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"\n  Steps/epoch (natural): {steps_per_epoch:,}")
print(f"  Total steps:   {total_steps:,}")
print(f"  Warmup steps:  {warmup_steps:,}")
print(f"  Eval every:    {cfg.eval_every_steps} steps")
print(f"  Save every:    {cfg.save_every_steps} steps")


# ═══ CELL 9 — Resume Support ════════════════════════════════════════════════

start_epoch  = 0
start_step   = 0
global_step  = 0
best_val_ppl = float("inf")
best_step    = 0
patience_ctr = 0

if os.path.exists(cfg.resume_state):
    print(f"\n  Found resume state at {cfg.resume_state} — loading...")
    state = torch.load(cfg.resume_state, map_location="cpu")
    try:
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        # Model LoRA weights loaded from best/final dir separately
        latest_ckpt = state.get("latest_ckpt")
        if latest_ckpt and os.path.exists(latest_ckpt):
            # Replace model's LoRA weights with the saved checkpoint
            from peft import set_peft_model_state_dict
            adapter_weights = torch.load(
                os.path.join(latest_ckpt, "adapter_model.bin"),
                map_location=device,
            ) if os.path.exists(os.path.join(latest_ckpt, "adapter_model.bin")) else None
            # safetensors variant
            if adapter_weights is None:
                try:
                    from safetensors.torch import load_file
                    adapter_weights = load_file(
                        os.path.join(latest_ckpt, "adapter_model.safetensors"),
                        device=str(device),
                    )
                except Exception:
                    pass
            if adapter_weights is not None:
                set_peft_model_state_dict(model, adapter_weights)
                print(f"  ✓ Loaded LoRA weights from {latest_ckpt}")
        start_step    = state["global_step"]
        global_step   = state["global_step"]
        best_val_ppl  = state.get("best_val_ppl", float("inf"))
        best_step     = state.get("best_step", 0)
        patience_ctr  = state.get("patience_ctr", 0)
        print(f"  ✓ Resumed at step {start_step} "
              f"(best_val_ppl={best_val_ppl:.3f} @ step {best_step})")
    except Exception as e:
        print(f"  ⚠ Resume failed ({e}) — starting fresh")
        start_step = 0
        global_step = 0


# ═══ CELL 10 — Eval Helper ══════════════════════════════════════════════════

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=compute_dtype):
            out = model(**batch)
        ntok = batch["attention_mask"].sum().item()
        total_loss += out.loss.item() * ntok
        total_tokens += ntok
    model.train()
    if total_tokens == 0:
        return float("inf"), float("inf")
    nll = total_loss / total_tokens
    ppl = math.exp(min(nll, 20))
    return nll, ppl


# ═══ CELL 11 — Training Loop ════════════════════════════════════════════════

print("\n" + "=" * 72)
print(f"TRAINING — {cfg.num_epochs} epoch over {n_train_tokens/1e9:.2f}B tokens")
print("=" * 72)

history = {
    "step": [], "train_loss": [], "val_nll": [], "val_ppl": [],
    "lr": [], "grad_norm": [], "tok_per_sec": [], "vram_gb": [],
    "elapsed_hr": [],
}
# Load prior history if resuming
prior_hist = os.path.join(cfg.log_dir, "history.json")
if start_step > 0 and os.path.exists(prior_hist):
    with open(prior_hist) as f:
        history = json.load(f)
    print(f"  Loaded {len(history['step'])} prior eval points")

running_losses = []
total_tokens_processed = 0
oom_count = 0
start_time = time.time()
should_stop = False


def save_lora(path):
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)


def save_resume_state(latest_ckpt_path):
    torch.save({
        "optimizer":    optimizer.state_dict(),
        "scheduler":    scheduler.state_dict(),
        "global_step":  global_step,
        "best_val_ppl": best_val_ppl,
        "best_step":    best_step,
        "patience_ctr": patience_ctr,
        "latest_ckpt":  latest_ckpt_path,
    }, cfg.resume_state)


def prune_old_ckpts():
    ck = sorted(Path(cfg.ckpt_dir).glob("step_*"))
    if len(ck) > cfg.keep_last_n_ckpts:
        for old in ck[:-cfg.keep_last_n_ckpts]:
            for p in old.rglob("*"):
                if p.is_file():
                    p.unlink()
            old.rmdir()


model.train()
print(f"\n  ━━ TRAINING STARTED {time.strftime('%Y-%m-%d %H:%M:%S')} ━━")
if start_step > 0:
    print(f"  Resuming from step {start_step}\n")

optimizer.zero_grad()
micro_step_counter = 0

for epoch in range(start_epoch, cfg.num_epochs):
    if should_stop:
        break

    # v2: random stride offset per epoch to diversify note-window alignment
    if cfg.randomize_window_offset:
        epoch_offset = random.Random(cfg.seed + epoch).randint(0, cfg.max_seq_len - 1)
        train_ds.set_offset(epoch_offset)
        print(f"\n  ╔══ EPOCH {epoch+1}/{cfg.num_epochs} (window offset={epoch_offset}) ══╗")
    else:
        print(f"\n  ╔══ EPOCH {epoch+1}/{cfg.num_epochs} ══╗")

    for batch_idx, batch in enumerate(train_loader):
        if should_stop:
            break

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        ntok = batch["attention_mask"].sum().item()
        total_tokens_processed += ntok

        try:
            with torch.autocast("cuda", dtype=compute_dtype):
                out = model(**batch)
            loss = out.loss / cfg.grad_accum_steps
            loss.backward()
        except torch.cuda.OutOfMemoryError:
            oom_count += 1
            print(f"  ⚠ OOM at micro-step {batch_idx} (total OOMs: {oom_count})")
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            # If many consecutive OOMs, bail — something structural is wrong
            if oom_count > 20:
                print("  ✋ Too many OOMs — stopping.")
                should_stop = True
            continue

        running_losses.append(out.loss.item())
        micro_step_counter += 1

        if micro_step_counter % cfg.grad_accum_steps == 0:
            grad_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                cfg.max_grad_norm,
            ).item()
            optimizer.step()
            scheduler.step()
            # LR floor
            for pg in optimizer.param_groups:
                pg["lr"] = max(pg["lr"], cfg.lr * cfg.min_lr_ratio)
            optimizer.zero_grad()
            global_step += 1

            # ── Log ──
            if global_step % cfg.log_every_steps == 0:
                window = cfg.log_every_steps * cfg.grad_accum_steps
                avg = float(np.mean(running_losses[-window:]))
                lr_now = optimizer.param_groups[0]["lr"]
                elapsed = time.time() - start_time
                tok_s = total_tokens_processed / max(1, elapsed)
                vram_now = (torch.cuda.memory_allocated() / 1e9
                            if torch.cuda.is_available() else 0.0)
                print(f"  Step {global_step:>5}/{total_steps} | "
                      f"loss={avg:.4f} | LR={lr_now:.2e} | "
                      f"‖∇‖={grad_norm:.3f} | VRAM={vram_now:.1f}GB | "
                      f"{tok_s:.0f} tok/s | {elapsed/3600:.2f}h")

            # ── Eval ──
            if global_step % cfg.eval_every_steps == 0:
                print(f"\n  ── Eval at step {global_step} ──")
                val_nll, val_ppl = evaluate(model, val_loader,
                                            max_batches=cfg.eval_max_batches_mid)
                avg_train = float(np.mean(running_losses[-cfg.eval_every_steps:]))
                lr_now = optimizer.param_groups[0]["lr"]
                vram_now = (torch.cuda.memory_allocated() / 1e9
                            if torch.cuda.is_available() else 0.0)
                elapsed = time.time() - start_time
                tok_s = total_tokens_processed / max(1, elapsed)

                history["step"].append(global_step)
                history["train_loss"].append(avg_train)
                history["val_nll"].append(val_nll)
                history["val_ppl"].append(val_ppl)
                history["lr"].append(lr_now)
                history["grad_norm"].append(grad_norm)
                history["tok_per_sec"].append(tok_s)
                history["vram_gb"].append(vram_now)
                history["elapsed_hr"].append(elapsed / 3600)

                improved = val_ppl < best_val_ppl
                if improved:
                    best_val_ppl = val_ppl
                    best_step = global_step
                    patience_ctr = 0
                    save_lora(cfg.best_dir)
                    print(f"  val_ppl={val_ppl:.3f} ★ NEW BEST ★ "
                          f"(train={avg_train:.4f})")
                    print(f"  ✓ Best saved → {cfg.best_dir}")
                else:
                    patience_ctr += 1
                    print(f"  val_ppl={val_ppl:.3f} "
                          f"(no improve {patience_ctr}/{cfg.early_stop_patience}, "
                          f"train={avg_train:.4f})")
                    # v3: only early-stop AFTER warmup completes — warmup evals
                    # are noisy and can otherwise kill a healthy run.
                    past_warmup = global_step > warmup_steps
                    if past_warmup and patience_ctr >= cfg.early_stop_patience:
                        print(f"  ✋ EARLY STOP — val ppl plateaued post-warmup")
                        should_stop = True
                    elif patience_ctr >= cfg.early_stop_patience:
                        print(f"  (ignoring patience during warmup)")
                        patience_ctr = 0

                with open(os.path.join(cfg.log_dir, "history.json"), "w") as f:
                    json.dump(history, f, indent=2)
                print()

            # ── Periodic checkpoint ──
            if global_step % cfg.save_every_steps == 0:
                ck_path = os.path.join(cfg.ckpt_dir, f"step_{global_step}")
                save_lora(ck_path)
                save_resume_state(ck_path)
                prune_old_ckpts()
                print(f"  ✓ Checkpoint → {ck_path}")

            # ── Wall clock ──
            if time.time() - start_time >= cfg.max_training_seconds:
                print(f"\n  ⏰ Wall-clock limit hit at step {global_step}")
                should_stop = True

            # ── Step cap (v3) — exit cleanly when schedule completes ──
            if global_step >= total_steps:
                print(f"\n  ✓ Reached target total_steps={total_steps} — "
                      f"ending training cleanly.")
                should_stop = True

    elapsed_hr = (time.time() - start_time) / 3600
    print(f"\n  ╚══ Epoch {epoch+1} done | {elapsed_hr:.2f}h ══╝")

# Final save + full eval
save_lora(cfg.final_dir)
print(f"\n  ✓ Final adapter → {cfg.final_dir}")

print("\n" + "━" * 60)
print("FINAL EVAL (full val set)")
print("━" * 60)
final_nll, final_ppl = evaluate(model, val_loader,
                                max_batches=cfg.eval_max_batches_final)
print(f"  Final val_nll: {final_nll:.4f} | val_ppl: {final_ppl:.3f}")

with open(os.path.join(cfg.log_dir, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

elapsed_min = (time.time() - start_time) / 60
print("\n" + "━" * 60)
print(f"TRAINING COMPLETE  {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Duration:      {elapsed_min:.1f} min ({elapsed_min/60:.2f} h)")
print(f"  Best val_ppl:  {best_val_ppl:.3f} at step {best_step}")
print(f"  Final val_ppl: {final_ppl:.3f}")
print(f"  Total tokens:  {total_tokens_processed:,}")
print(f"  Throughput:    {total_tokens_processed/max(1, elapsed_min*60):.0f} tok/s")
print(f"  OOM skips:     {oom_count}")
print("━" * 60)


# ═══ CELL 12 — Plots ════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("GENERATING PLOTS")
print("=" * 72)

h = history
if len(h["step"]) >= 2:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    ax = axes[0, 0]
    ax.plot(h["step"], h["train_loss"], lw=2, label="Train", color="tab:blue")
    ax.plot(h["step"], h["val_nll"],   lw=2, label="Val",   color="tab:red",
            marker="o", ms=4)
    if best_step > 0:
        ax.axvline(best_step, ls="--", color="green", alpha=.6,
                   label=f"Best @ {best_step}")
    ax.set_xlabel("Step"); ax.set_ylabel("NLL")
    ax.set_title("Loss Curves"); ax.legend(); ax.grid(True, alpha=.3)

    ax = axes[0, 1]
    ax.plot(h["step"], h["val_ppl"], lw=2, marker="s", ms=4, color="tab:purple")
    ax.set_xlabel("Step"); ax.set_ylabel("Perplexity")
    ax.set_title("Held-Out Val Perplexity")
    ax.grid(True, alpha=.3)

    ax = axes[0, 2]
    ax.plot(h["step"], h["lr"], lw=2, color="tab:green")
    ax.set_xlabel("Step"); ax.set_ylabel("LR")
    ax.set_title("Learning Rate"); ax.grid(True, alpha=.3)
    ax.ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))

    ax = axes[1, 0]
    ax.plot(h["step"], h["grad_norm"], lw=1.5, color="tab:orange", alpha=.8)
    ax.set_xlabel("Step"); ax.set_ylabel("‖∇‖")
    ax.set_title("Gradient Norm"); ax.grid(True, alpha=.3)

    ax = axes[1, 1]
    ax.plot(h["step"], h["tok_per_sec"], lw=2, color="tab:cyan")
    ax.set_xlabel("Step"); ax.set_ylabel("tokens/sec")
    ax.set_title("Throughput"); ax.grid(True, alpha=.3)

    ax = axes[1, 2]
    ax.plot(h["step"], h["vram_gb"], lw=2, color="tab:brown")
    ax.axhline(vram_gb, ls="--", color="gray", alpha=.4,
               label=f"Device: {vram_gb:.0f} GB")
    ax.set_xlabel("Step"); ax.set_ylabel("VRAM (GB)")
    ax.set_title("VRAM Usage"); ax.legend(); ax.grid(True, alpha=.3)

    plt.tight_layout()
    plot_path = os.path.join(cfg.plot_dir, "training_curves.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {plot_path}")
else:
    print("  Not enough eval points.")


# ═══ CELL 13 — Sanity Generation ════════════════════════════════════════════

print("\n" + "=" * 72)
print("SANITY GENERATION (best checkpoint)")
print("=" * 72)

del model
force_gc("model deleted for reload")

base = AutoModelForCausalLM.from_pretrained(
    cfg.model_name, device_map="auto", trust_remote_code=True,
    torch_dtype=compute_dtype, attn_implementation=attn_impl,
)
model = PeftModel.from_pretrained(base, cfg.best_dir)
model.eval()

prompts = [
    "[DISCHARGE | 2150-06-01 12:00]\nChief Complaint: chest pain\n\nHistory of Present Illness: ",
    "[RADIOLOGY | 2150-06-02 09:30]\nEXAMINATION: CHEST X-RAY\nINDICATION: shortness of breath\nFINDINGS: ",
    "[DISCHARGE | 2150-06-03 08:00]\nBrief Hospital Course:\nThe patient was admitted for ",
]
samples = []
for i, prompt in enumerate(prompts):
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **ids, max_new_tokens=200, do_sample=True,
            temperature=0.7, top_p=0.9, repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = tokenizer.decode(out[0][ids["input_ids"].shape[-1]:],
                           skip_special_tokens=True)
    samples.append({"prompt": prompt, "generation": gen})
    print(f"\n  --- Sample {i+1} ---")
    print(f"  PROMPT: {prompt[:120]}...")
    print(f"  GEN:    {gen[:300]}...")

with open(os.path.join(cfg.log_dir, "generation_samples.json"), "w") as f:
    json.dump(samples, f, indent=2, default=str)


# ═══ CELL 14 — Final Summary ════════════════════════════════════════════════

summary = {
    "version":          "CPT-v3",
    "model":            cfg.model_name,
    "gpu":              gpu_name,
    "vram_gb":          vram_gb,
    "train_tokens":     n_train_tokens,
    "val_tokens":       n_val_tokens,
    "train_notes":      meta["n_train_notes"],
    "val_notes":        meta["n_val_notes"],
    "max_seq_len":      cfg.max_seq_len,
    "effective_batch":  cfg.batch_size * cfg.grad_accum_steps,
    "batch_size":       cfg.batch_size,
    "grad_accum":       cfg.grad_accum_steps,
    "lora_r":           cfg.lora_r,
    "lora_alpha":       cfg.lora_alpha,
    "lora_dropout":     cfg.lora_dropout,
    "trainable_params": trainable,
    "total_params":     total_p,
    "attention":        attn_impl,
    "compute_dtype":    str(compute_dtype),
    "epochs":           cfg.num_epochs,
    "lr":               cfg.lr,
    "best_val_ppl":     float(best_val_ppl),
    "best_step":        int(best_step),
    "final_val_ppl":    float(final_ppl),
    "final_val_nll":    float(final_nll),
    "total_steps":      int(global_step),
    "total_tokens":     int(total_tokens_processed),
    "training_min":     round(elapsed_min, 1),
    "avg_tok_per_sec":  round(total_tokens_processed/max(1, elapsed_min*60), 1),
    "oom_skips":        oom_count,
    "paths": {
        "best_model":  cfg.best_dir,
        "final_model": cfg.final_dir,
        "checkpoints": cfg.ckpt_dir,
        "plots":       cfg.plot_dir,
        "logs":        cfg.log_dir,
        "cache":       cfg.cache_dir,
    },
    "v3_changes_vs_v2": [
        "grad_accum 24→8 (3× more optim steps per unit time)",
        "max_optimizer_steps=900 (cosine fully anneals in 5h budget)",
        "warmup_ratio 0.03→0.05 (faster warmup on short run)",
        "eval_every 200→50, save_every 1000→150 (tighter tracking)",
        "wall-clock 22h→4.5h (hard guarantee w/ buffer for final eval)",
        "early-stop patience 5→4 (bail faster)",
        "kept from v2: batch=2, lora_r=16, dropout=0.10, flash-attn assert",
    ],
}
with open(os.path.join(cfg.log_dir, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 72)
print("FINAL SUMMARY")
print("=" * 72)
print(json.dumps(summary, indent=2, default=str))

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║  BioMistral-7B CPT v2 — Complete                                     ║
║                                                                      ║
║  Outputs at {cfg.output_dir}/:
║    best_model/    — LoRA adapter (best val_ppl, early-stop winner)
║    final_model/   — last-checkpoint adapter
║    checkpoints/   — rotating step-N (last {cfg.keep_last_n_ckpts})
║    plots/         — 6-panel training diagnostics
║    logs/          — history.json, training_summary.json, gen samples
║    cache/         — packed token memmaps (reused across runs)
║
║  Load for TEGoT:                                                     ║
║    base  = AutoModelForCausalLM.from_pretrained('{cfg.model_name}')
║    model = PeftModel.from_pretrained(base, '{cfg.best_dir}')
╚══════════════════════════════════════════════════════════════════════╝
""")


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✓ Dependencies ready.
BioMistral-7B CPT on MIMIC-IV Notes — v3 (5-hour deadline build)
  Device:          cuda
  GPU:             NVIDIA H200 | VRAM: 150.1 GB
  Output:          /mnt/biomistral
  Model:           BioMistral/BioMistral-7B
  LoRA:            r=16, alpha=32, dropout=0.1 (anti-overfit)
  Seq len:         4096  (packed)
  Batch:           2 × 8 accum = 16 effective
  LR:              2.0e-04 cosine → 1.0e-05
  Early stop:      patience=4 evals on val_ppl
  Wall-clock cap:  4 h

LOADING TOKENIZER
  Vocab: 32,000  BOS=1 EOS=2 PAD=2

  ✓ Token cache found — skipping rebuild.

  Using cache: train=2092.4M tok, val=10.92M tok, train_notes=2,608,855, val_notes=13,339

  Packed train windows: 510,830
  Packed val windows:   2,665

LOADING BioMistral-7B (bf16 + Flash Attention 2 + LoRA r=16)
  Compute dtype: torch.bfloat16
  ✓ flash_attn 2.8.3 imported
  Attention:     flash_attention_2 (forced)
  config._attn_implementation = flash_attention_2
  Base loaded in 8.2s
  [VRAM] after 

In [11]:
from huggingface_hub import HfApi, create_repo, upload_folder

repo_id = "GOVINDFROM/BioMistral-7B-MIMIC-Notes"
folder_path = "/mnt/biomistral"  # contains model + plots

# Create repo (if not already created)
create_repo(repo_id, repo_type="model", exist_ok=True)

# Upload entire folder
upload_folder(
    repo_id=repo_id,
    folder_path=folder_path,
    path_in_repo="",  # root of repo
)

In [15]:
# ═══ POST-RUN DIAGNOSTIC — what did v3 actually produce? ═══
import os, json
from pathlib import Path

OUT = "/mnt/biomistral"

# 1. Training history — the actual val_ppl trajectory
hist_path = f"{OUT}/logs/history.json"
if os.path.exists(hist_path):
    with open(hist_path) as f:
        h = json.load(f)
    print("=" * 70)
    print("TRAINING HISTORY — full val_ppl trajectory from v3 run")
    print("=" * 70)
    print(f"  Total eval points: {len(h['step'])}")
    print(f"  Final step:        {h['step'][-1] if h['step'] else 'N/A'}")
    print(f"  Lowest val_ppl:    {min(h['val_ppl']):.4f} at step "
          f"{h['step'][h['val_ppl'].index(min(h['val_ppl']))]}")
    print(f"  Last val_ppl:      {h['val_ppl'][-1]:.4f}")
    print(f"  Last train_loss:   {h['train_loss'][-1]:.4f}")
    print(f"\n  Full curve:")
    print(f"  {'step':>6} {'val_ppl':>9} {'train':>9} {'LR':>10}")
    for i in range(len(h["step"])):
        print(f"  {h['step'][i]:>6} {h['val_ppl'][i]:>9.4f} "
              f"{h['train_loss'][i]:>9.4f} {h['lr'][i]:>10.2e}")
else:
    print(f"  ✗ No history at {hist_path}")

# 2. What checkpoints / models exist
print("\n" + "=" * 70)
print("WHAT'S SAVED")
print("=" * 70)
for d in ["best_model", "final_model", "checkpoints"]:
    p = Path(OUT) / d
    if p.exists():
        if d == "checkpoints":
            subs = sorted([x.name for x in p.iterdir() if x.is_dir()])
            files = sorted([x.name for x in p.iterdir() if x.is_file()])
            print(f"  {d}/  subdirs: {subs}")
            print(f"  {d}/  files:   {files}")
        else:
            files = [x.name for x in p.iterdir()]
            has_adapter = any("adapter_model" in f for f in files)
            print(f"  {d}/: has_adapter={has_adapter}, files={len(files)}")
    else:
        print(f"  {d}/: MISSING")

# 3. Read training_summary.json if it exists
print("\n" + "=" * 70)
print("TRAINING SUMMARY (if written)")
print("=" * 70)
summary_path = f"{OUT}/logs/training_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        s = json.load(f)
    for k in ["version", "best_val_ppl", "best_step", "final_val_ppl",
              "total_steps", "training_min", "avg_tok_per_sec"]:
        if k in s:
            print(f"  {k:25s}: {s[k]}")
else:
    print(f"  ✗ No summary at {summary_path}")
    print("  (This means the cell was interrupted before STAGE 4-5,")
    print("   so final full-val eval + plots didn't run.)")

# 4. Plot existence
plot_path = f"{OUT}/plots/training_curves.png"
print(f"\n  plots/training_curves.png: "
      f"{'EXISTS' if os.path.exists(plot_path) else 'MISSING'}")